# 03 — Incertidumbre de observación adaptativa

> Esta comparación pregunta si una incertidumbre de observación dinámica cambia materialmente la atribución de Kalman. El periodo 2022–2026 ya fue explorado: los resultados son causales y descriptivos, no confirmatorios.

## <b>Modelos y cronología</b>

Se comparan rolling OLS de 252 sesiones, Kalman con $Q/R$ fijos y Kalman con $Q$ fijo y $R_t$ adaptativo. Los tres usan QQQ, XLE y XLF, MKT/SIZE/GV, retornos diarios Gold y las fechas de evaluación de la notebook 02.

En el modelo adaptativo, al inicio de $t$ se usa $R_t$ calculado sólo con innovaciones hasta $t-1$. Tras atribuir $R_t$, se actualiza $R_{t+1} = max(R_{floor}, 0.94R_t + 0.06e_t^2)$. La innovación también contiene incertidumbre de beta: este EWMA aproxima incertidumbre de observación, no mide ruido residual puro.

In [1]:
from datetime import date
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import polars as pl
from plotly.subplots import make_subplots
from IPython.display import HTML, display

TARGETS = ('QQQ', 'XLE', 'XLF')
SYMBOLS = ('SPY', 'QQQ', 'IWM', 'IWD', 'IWF', 'XLE', 'XLF')
FACTORS = ('market', 'size', 'growth_value')
RESEARCH_START, RESEARCH_END = date(2018, 1, 2), date(2026, 11, 17)
CALIBRATION_END, EVALUATION_START = date(2021, 12, 31), date(2022, 1, 1)
ROLLING_WINDOW, LAMBDA = 252, 0.94

def project_root(start):
    for parent in (start.resolve(), *start.resolve().parents):
        direct = parent / 'Momentum_systematic_strategies'
        if (direct / 'data' / 'gold' / 'daily_series' / 'etf').is_dir():
            return direct
        if (parent / 'data' / 'gold' / 'daily_series' / 'etf').is_dir():
            return parent
    raise FileNotFoundError('No se encontró data/gold/daily_series/etf.')

GOLD = project_root(Path.cwd()) / 'data' / 'gold' / 'daily_series' / 'etf'
frames = {symbol: pl.read_parquet(GOLD / f'{symbol}.parquet').with_columns(pl.col('session_date').cast(pl.Date)).filter(
    pl.col('session_date').is_between(RESEARCH_START, RESEARCH_END, closed='both')).sort('session_date') for symbol in SYMBOLS}
prices = frames['SPY'].select(pl.col('session_date').alias('date'), pl.col('adjusted_close').alias('SPY'))
for symbol in SYMBOLS[1:]:
    prices = prices.join(frames[symbol].select(pl.col('session_date').alias('date'), pl.col('adjusted_close').alias(symbol)), on='date', how='inner')
returns = prices.sort('date').with_columns([pl.col(symbol).log().diff().alias(f'{symbol}_ret') for symbol in SYMBOLS]).drop_nulls()
model_panel = returns.select('date', *[f'{asset}_ret' for asset in TARGETS], 'SPY_ret', 'IWM_ret', 'IWF_ret', 'IWD_ret').with_columns(
    pl.col('SPY_ret').alias('market'), (pl.col('IWM_ret') - pl.col('SPY_ret')).alias('size'),
    (pl.col('IWF_ret') - pl.col('IWD_ret')).alias('growth_value')).select('date', *[f'{asset}_ret' for asset in TARGETS], *FACTORS).drop_nulls()
dates, factor_values = model_panel.get_column('date').to_list(), model_panel.select(*FACTORS).to_numpy()
evaluation_index = next(index for index, value in enumerate(dates) if value >= EVALUATION_START)
assert dates[evaluation_index - 1] <= CALIBRATION_END

## <b>Calibración y benchmark congelados</b>

La calibración reproduce la notebook 02: $R$ es la varianza de los errores OOS rolling disponibles hasta 2021, y $Q$ es diagonal con la varianza de los cambios diarios de los coeficientes rolling del mismo periodo. El piso es $R_{floor}=0.01R_{calibration}$: una regla fija, no optimizada, determinada antes de evaluación.

In [2]:
ols_rows, calibration = [], {}
for asset in TARGETS:
    actual = model_panel.get_column(f'{asset}_ret').to_numpy()
    history, residual_history = [], []
    for index in range(ROLLING_WINDOW, len(actual)):
        design = np.column_stack((np.ones(ROLLING_WINDOW), factor_values[index - ROLLING_WINDOW:index]))
        beta = np.linalg.lstsq(design, actual[index - ROLLING_WINDOW:index], rcond=None)[0]
        predicted = np.r_[1.0, factor_values[index]] @ beta
        residual = actual[index] - predicted
        ols_rows.append({'date': dates[index], 'beta_asof_date': dates[index - 1], 'asset': asset,
                         'alpha': beta[0], 'beta_market': beta[1], 'beta_size': beta[2], 'beta_growth_value': beta[3],
                         'actual_return': actual[index], 'predicted_return': predicted, 'residual_return': residual})
        if dates[index] <= CALIBRATION_END:
            history.append(beta)
            residual_history.append(residual)
    history = np.asarray(history)
    r = float(np.var(residual_history, ddof=1))
    calibration[asset] = {
        'Q': np.diag(np.var(np.diff(history, axis=0), axis=0, ddof=1)), 'R': r, 'R_floor': max(np.finfo(float).eps, 0.01 * r),
        'theta': history[-1],
        'P': r * np.linalg.pinv(np.column_stack((np.ones(ROLLING_WINDOW), factor_values[evaluation_index - ROLLING_WINDOW:evaluation_index])).T @
                              np.column_stack((np.ones(ROLLING_WINDOW), factor_values[evaluation_index - ROLLING_WINDOW:evaluation_index]))),
    }
ols = pl.DataFrame(ols_rows)
calibration_summary = pl.DataFrame([{'asset': asset, 'R_fixed': values['R'], 'R_floor': values['R_floor'],
                                         'Q_MKT': values['Q'][1, 1], 'Q_SIZE': values['Q'][2, 2], 'Q_GV': values['Q'][3, 3]}
                                        for asset, values in calibration.items()]).with_columns(pl.all().exclude('asset').round(10))
calibration_summary

asset,R_fixed,R_floor,Q_MKT,Q_SIZE,Q_GV
str,f64,f64,f64,f64,f64
"""QQQ""",0.000006,5.7400e-8,0.000005,0.00001,0.000017
"""XLE""",0.000188,0.000002,0.000216,0.00032,0.00054
"""XLF""",0.000027,2.7410e-7,0.000025,0.000053,0.000061


## <b>Filtros causales</b>

El Kalman fijo reproduce la especificación anterior. El adaptativo conserva $Q$, estado inicial y $P$; sólo sustituye la varianza observacional usada al inicio de cada sesión. La actualización de $R$ ocurre después de calcular la innovación de esa sesión.

In [3]:
def run_kalman(asset, adaptive):
    actual = model_panel.get_column(f'{asset}_ret').to_numpy()
    theta, covariance = calibration[asset]['theta'].copy(), calibration[asset]['P'].copy()
    q, r_fixed, r_floor = calibration[asset]['Q'], calibration[asset]['R'], calibration[asset]['R_floor']
    r_current, rows = r_fixed, []
    for index in range(evaluation_index, len(actual)):
        x = np.r_[1.0, factor_values[index]]
        theta_prior, covariance_prior = theta, covariance + q
        predicted, error = x @ theta_prior, actual[index] - x @ theta_prior
        r_used = r_current if adaptive else r_fixed
        gain = covariance_prior @ x / float(x @ covariance_prior @ x + r_used)
        theta = theta_prior + gain * error
        update = np.eye(4) - np.outer(gain, x)
        covariance = update @ covariance_prior @ update.T + np.outer(gain, gain) * r_used
        rows.append({'date': dates[index], 'beta_asof_date': dates[index - 1], 'asset': asset,
                     'alpha': theta_prior[0], 'beta_market': theta_prior[1], 'beta_size': theta_prior[2],
                     'beta_growth_value': theta_prior[3], 'actual_return': actual[index],
                     'predicted_return': predicted, 'residual_return': error, 'R_used': r_used})
        if adaptive:
            r_current = max(r_floor, LAMBDA * r_current + (1 - LAMBDA) * error ** 2)
    return pl.DataFrame(rows)

kalman_fixed = pl.concat([run_kalman(asset, adaptive=False) for asset in TARGETS])
kalman_adaptive = pl.concat([run_kalman(asset, adaptive=True) for asset in TARGETS])
ols_evaluation = ols.filter(pl.col('date') >= EVALUATION_START)
causality_audit = pl.DataFrame([{
    'ols_rows': ols_evaluation.height, 'fixed_rows': kalman_fixed.height, 'adaptive_rows': kalman_adaptive.height,
    'adaptive_betas_precede_return': (kalman_adaptive.get_column('beta_asof_date') < kalman_adaptive.get_column('date')).all(),
    'common_dates': ols_evaluation.select('date', 'asset').join(kalman_adaptive.select('date', 'asset'), on=['date', 'asset']).height,
}])
causality_audit

ols_rows,fixed_rows,adaptive_rows,adaptive_betas_precede_return,common_dates
i64,i64,i64,bool,i64
3672,3672,3672,true,3672


## <b>Comparación de atribución</b>

Las métricas usan exactamente las mismas 1,224 fechas por activo. La variabilidad de beta es su desviación estándar media entre MKT, SIZE y GV; el cambio diario es la media de los movimientos absolutos de esas tres betas.

In [4]:
frames_by_model = (('Rolling OLS 252d', ols_evaluation), ('Kalman fixed R', kalman_fixed), ('Kalman adaptive R', kalman_adaptive))
summary_rows, subperiod_rows = [], []
for model, frame in frames_by_model:
    for asset in TARGETS:
        sample = frame.filter(pl.col('asset') == asset).sort('date')
        actual, error = sample.get_column('actual_return').to_numpy(), sample.get_column('residual_return').to_numpy()
        betas = sample.select('beta_market', 'beta_size', 'beta_growth_value').to_numpy()
        summary_rows.append({'asset': asset, 'model': model, 'n': sample.height, 'mae_pct': np.abs(error).mean() * 100,
                             'rmse_pct': np.sqrt(np.mean(error ** 2)) * 100,
                             'oos_r2': 1 - (error ** 2).sum() / ((actual - actual.mean()) ** 2).sum(),
                             'p95_abs_error_pct': np.quantile(np.abs(error), .95) * 100,
                             'beta_std_mean': betas.std(axis=0, ddof=1).mean(),
                             'beta_mean_abs_daily_change': np.abs(np.diff(betas, axis=0)).mean()})
        for label, start, end in (('2022', date(2022, 1, 1), date(2022, 12, 31)), ('2023–2026', date(2023, 1, 1), RESEARCH_END)):
            period = sample.filter(pl.col('date').is_between(start, end, closed='both'))
            period_error = period.get_column('residual_return').to_numpy()
            subperiod_rows.append({'asset': asset, 'model': model, 'period': label, 'n': period.height,
                                   'mae_pct': np.abs(period_error).mean() * 100, 'rmse_pct': np.sqrt(np.mean(period_error ** 2)) * 100,
                                   'p95_abs_error_pct': np.quantile(np.abs(period_error), .95) * 100})
comparison_summary = pl.DataFrame(summary_rows).with_columns(pl.all().exclude('asset', 'model', 'n').round(4))
fixed_reference_r2 = {'QQQ': 0.9708, 'XLE': 0.3351, 'XLF': 0.7792}
fixed_reproduction = comparison_summary.filter(pl.col('model') == 'Kalman fixed R').select('asset', 'oos_r2').with_columns(
    pl.Series('reference_r2', [fixed_reference_r2[asset] for asset in TARGETS]),
).with_columns((pl.col('oos_r2') - pl.col('reference_r2')).abs().alias('abs_difference'))
if fixed_reproduction.get_column('abs_difference').max() > 0.0001:
    raise AssertionError('Kalman fixed R no reproduce la notebook 02.')
subperiod_summary = pl.DataFrame(subperiod_rows).with_columns(pl.all().exclude('asset', 'model', 'period', 'n').round(4))
fixed_reproduction, comparison_summary, subperiod_summary

(shape: (3, 4)
 ┌───────┬────────┬──────────────┬────────────────┐
 │ asset ┆ oos_r2 ┆ reference_r2 ┆ abs_difference │
 │ ---   ┆ ---    ┆ ---          ┆ ---            │
 │ str   ┆ f64    ┆ f64          ┆ f64            │
 ╞═══════╪════════╪══════════════╪════════════════╡
 │ QQQ   ┆ 0.9708 ┆ 0.9708       ┆ 0.0            │
 │ XLE   ┆ 0.3351 ┆ 0.3351       ┆ 0.0            │
 │ XLF   ┆ 0.7792 ┆ 0.7792       ┆ 0.0            │
 └───────┴────────┴──────────────┴────────────────┘,
 shape: (9, 9)
 ┌───────┬───────────────┬──────┬─────────┬───┬────────┬──────────────┬──────────────┬──────────────┐
 │ asset ┆ model         ┆ n    ┆ mae_pct ┆ … ┆ oos_r2 ┆ p95_abs_erro ┆ beta_std_mea ┆ beta_mean_ab │
 │ ---   ┆ ---           ┆ ---  ┆ ---     ┆   ┆ ---    ┆ r_pct        ┆ n            ┆ s_daily_chan │
 │ str   ┆ str           ┆ i64  ┆ f64     ┆   ┆ f64    ┆ ---          ┆ ---          ┆ ge           │
 │       ┆               ┆      ┆         ┆   ┆        ┆ f64          ┆ f64          ┆ ---   

## <b>Incertidumbre de XLE y episodios de error elevado</b>

> Los episodios se eligen mecánicamente como los tres mayores errores absolutos de rolling OLS de XLE, separados por al menos 21 sesiones. La regla no mira cuál modelo resulta favorecido. Cada panel muestra diez sesiones antes y después del episodio.

In [6]:
xle_fixed = kalman_fixed.filter(pl.col('asset') == 'XLE').sort('date')
xle_adaptive = kalman_adaptive.filter(pl.col('asset') == 'XLE').sort('date')
r_fixed = calibration['XLE']['R']
figure = go.Figure()
figure.add_scatter(x=xle_adaptive['date'], y=xle_adaptive['R_used'], mode='lines', name='Adaptive R')
figure.add_hline(y=r_fixed, line_dash='dash', line_color='gray', annotation_text='Fixed R')
figure.update_layout(title='XLE — observation uncertainty used before each return', xaxis_title='return date', yaxis_title='R', template='plotly_white')
display(HTML(figure.to_html(include_plotlyjs='cdn', full_html=False)))


 

In [7]:
xle_ols = ols_evaluation.filter(pl.col('asset') == 'XLE').sort('date')
ordered = np.argsort(-np.abs(xle_ols.get_column('residual_return').to_numpy()))
episode_index = []
for raw_index in ordered:
    index = int(raw_index)
    if all(abs(index - chosen) > 21 for chosen in episode_index):
        episode_index.append(index)
    if len(episode_index) == 3:
        break
episode_index.sort()
figure = make_subplots(rows=1, cols=3, subplot_titles=[str(xle_ols.get_column('date')[index]) for index in episode_index])
for column, center in enumerate(episode_index, start=1):
    start, end = max(0, center - 10), min(xle_ols.height, center + 11)
    for label, frame in (('Rolling OLS', xle_ols), ('Kalman fixed R', xle_fixed), ('Kalman adaptive R', xle_adaptive)):
        sample = frame.slice(start, end - start)
        figure.add_scatter(x=sample['date'], y=sample['residual_return'] * 100, mode='lines+markers', name=label,
                           legendgroup=label, showlegend=column == 1, row=1, col=column)
figure.update_layout(title='XLE — attribution errors around mechanically selected OLS episodes', yaxis_title='error (%)', template='plotly_white', height=380)
display(HTML(figure.to_html(include_plotlyjs='cdn', full_html=False)))

## <b>Conclusión</b>

- Frente al Kalman fijo, el $R$ adaptativo no mejora consistentemente MAE, RMSE o $R^2$. En XLE el $R^2$ baja levemente de 0.3351 a 0.3344.
- Sólo XLE reduce de forma marginal el p95 de error frente al Kalman fijo (2.6863% a 2.6750%). QQQ y XLF empeoran en ese extremo.
- El modelo adaptativo reduce la dispersión media de betas en los tres activos, incluido XLE (0.2869 a 0.2747), pero sus cambios diarios no disminuyen; en XLE aumentan de 0.0102 a 0.0113.
- En XLE, la mejora marginal del extremo no compensa evidencia clara de adaptación superior: en 2023–2026 el Kalman fijo conserva menor MAE y RMSE. Los gráficos muestran respuesta a shocks, no una prueba de que evite sobrerreacción.
- Con lambda, $Q$ y el piso fijados antes de evaluación, la ganancia es marginal y no justifica aún mayor complejidad. El EWMA no separa ruido observacional, exposición omitida y beta desactualizada; tampoco prueba predicción, alpha ni una estrategia operable.